## Telugu Manual Datasets — Phase 1 Summary

All sources use raw Wikimedia XML dumps, Internet Archive OCR text, or direct web scraping.
**No Hugging Face datasets used here.**

### Phase 1a — Completed (80.98M words)

| File | Lines | Words | Source |
| :--- | :---: | :---: | :--- |
| `telugu_wikipedia.jsonl` | 87,853 | 35,126,665 | `dumps.wikimedia.org/tewiki/latest/` XML dump |
| `telugu_wikisource_manual.jsonl` | 150,778 | 26,117,478 | `dumps.wikimedia.org/tewikisource/latest/` XML dump |
| `telugu_wikibooks.jsonl` | 1,589 | 3,287,289 | `dumps.wikimedia.org/tewikibooks/latest/` XML dump |
| `telugu_wikiquote.jsonl` | 1,505 | 221,777 | `dumps.wikimedia.org/tewikiquote/latest/` XML dump |
| `telugu_wiktionary.jsonl` | 93,708 | 4,543,799 | `dumps.wikimedia.org/tewiktionary/latest/` XML dump |
| `telugu_vipula_magazine.jsonl` | 289,579 | 7,160,748 | Internet Archive `vipula-eenadu-group` — 306 issues |
| `telugu_newspapers.jsonl` | 111,268 | 4,063,244 | Internet Archive — Andhra Jyothi, Eenadu, Sakshi |
| `telugu_vignana_sarvasvam.jsonl` | 6,262 | 321,407 | Internet Archive — Telugu encyclopedia |
| `telugu_puranas_classical.jsonl` | 2,294 | 111,064 | Internet Archive — Brahmanda Puranam, Visnu Puranam |
| `bharatiya_chitrakala_ocr.jsonl` | 229 | 21,593 | Local Tesseract OCR — *Bharatiya Chitrakala* |
| `navarasa_kadambari_ocr.jsonl` | 25 | 2,568 | Local Tesseract OCR — *Navarasa Kadambari* |
| **Phase 1a TOTAL** | | **80,977,632** | |

### Phase 1b — Newly Scraped (30.15M words)

| File | Lines | Words | Source |
| :--- | :---: | :---: | :--- |
| `telugu_ia_books.jsonl` | 368,193 | 28,355,416 | Internet Archive `language:Telugu` — djvu.txt bulk |
| `telugu_news_eenadu.jsonl` | 0 | 0 | eenadu.net — partial recent scrape |
| `telugu_news_andhrajyothy.jsonl` | 5,530 | 1,790,143 | andhrajyothy.com — categories & districts |
| **Phase 1b TOTAL** | | **30,145,559** | |
| **GRAND TOTAL** | | **111,123,191** | |

## 1. Imports & Setup

Run this cell first. It installs dependencies and sets the output directory.

In [ ]:
import os
import io
import re
import bz2
import json
import time
import ssl
import gzip
import shutil
import subprocess
import threading
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from pathlib import Path
from queue import Queue

# Optional — only needed for PDF OCR cells
try:
    import requests
    import pymupdf
    import pytesseract
    from PIL import Image
    from tqdm import tqdm
except ImportError:
    pass  # Not required if OCR cells are skipped

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_MANUAL_DIR = Path("../datasets/raw/manual/")
RAW_MANUAL_DIR.mkdir(parents=True, exist_ok=True)

# ── Shared helpers ─────────────────────────────────────────────────────────
_ctx = ssl._create_unverified_context()
_telugu_re = re.compile(r'[\u0C00-\u0C7F]')

def _ia_metadata(item_id):
    """Return list of (item_id, filename) tuples for all .txt files in an IA item."""
    url = f"https://archive.org/metadata/{item_id}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=_ctx, timeout=20) as r:
        data = json.loads(r.read())
    return [(item_id, f["name"]) for f in data.get("files", []) if f.get("name", "").endswith(".txt")]

def _ia_fetch(item_id, filename):
    """Download one file from the Internet Archive with retry."""
    url = f"https://archive.org/download/{item_id}/{urllib.parse.quote(filename)}"
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=_ctx, timeout=45) as r:
                return r.read().decode("utf-8", errors="ignore")
        except Exception:
            time.sleep(2 ** attempt)
    return ""

def _ia_download_parallel(files, out_path, source_tag, n_threads=10):
    """Download a list of (item_id, filename) pairs in parallel, save paragraphs to JSONL."""
    q = Queue()
    lock = threading.Lock()
    saved = [0]
    for item in files:
        q.put(item)

    def _worker():
        while not q.empty():
            try:
                item_id, fname = q.get_nowait()
            except Exception:
                break
            content = _ia_fetch(item_id, fname)
            rows = [
                json.dumps({"text": p.strip(), "source": source_tag}, ensure_ascii=False)
                for p in content.split("\n\n")
                if len(p.strip()) > 60 and _telugu_re.search(p)
            ]
            if rows:
                with lock:
                    with open(out_path, "a", encoding="utf-8") as fh:
                        fh.write("\n".join(rows) + "\n")
                    saved[0] += len(rows)
            q.task_done()

    threads = [threading.Thread(target=_worker) for _ in range(n_threads)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return saved[0]

def _parse_wiki_xml(dump_compressed, output_file, source_tag, skip_prefixes):
    """Parse a bz2-compressed Wikimedia XML dump and write paragraphs to JSONL."""
    ns_prefix = ""
    with bz2.open(dump_compressed, "rt", encoding="utf-8") as f:
        first_line = f.readline()
        m = re.search(r'xmlns="([^"]+)"', first_line)
        if m:
            ns_prefix = f"{{{m.group(1)}}}"
    print(f"  Namespace: {ns_prefix!r}")

    pages_saved = 0
    with open(output_file, "w", encoding="utf-8") as out:
        with bz2.open(dump_compressed, "rt", encoding="utf-8") as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                if elem.tag == f"{ns_prefix}page":
                    title_el = elem.find(f"{ns_prefix}title")
                    ns_el    = elem.find(f"{ns_prefix}ns")
                    rev_el   = elem.find(f"{ns_prefix}revision")
                    title = title_el.text if title_el is not None else ""
                    ns    = ns_el.text    if ns_el    is not None else ""
                    if ns == "0" and title and not any(title.startswith(p) for p in skip_prefixes):
                        text_el = rev_el.find(f"{ns_prefix}text") if rev_el is not None else None
                        text = text_el.text if text_el is not None else ""
                        if text and len(text) > 150:
                            text = re.sub(r'\{\{[^}]*\}\}', '', text)
                            text = re.sub(r'\[\[[^\]]*\]\]', '', text)
                            text = re.sub(r'<[^>]*>', '', text)
                            text = text.strip()
                            if len(text) > 150:
                                out.write(json.dumps({"text": text, "title": title, "source": source_tag},
                                                      ensure_ascii=False) + "\n")
                                pages_saved += 1
                    elem.clear()
    return pages_saved

print("Setup complete. RAW_MANUAL_DIR =", RAW_MANUAL_DIR.resolve())

## 2. Telugu Wikipedia XML Dump (`telugu_wikipedia.jsonl`)

Download and parse the official Telugu Wikipedia dump from `dumps.wikimedia.org`.

In [ ]:
output_file    = RAW_MANUAL_DIR / "telugu_wikipedia.jsonl"
dump_url       = "https://dumps.wikimedia.org/tewiki/latest/tewiki-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "tewiki-latest-pages-articles.xml.bz2"
te_skip = ["మూస:", "వర్గం:", "చిత్రం:", "వికీపీడియా:", "సహాయం:", "వాడుకరి:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print(f"Downloading Wikipedia dump (~900 MB)...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
        print("Download complete.")
    print("Parsing XML dump...")
    n = _parse_wiki_xml(dump_compressed, output_file, "telugu_wikipedia_dump", te_skip)
    print(f"Done. Saved {n:,} articles → {output_file.name}")

## 3. Local Tesseract OCR Pipeline

Download classical scanned PDFs and run local OCR page-by-page. Requires `tesseract` with `tel` language pack and `pymupdf`, `pytesseract`, `Pillow` installed.

In [ ]:
def process_pdf_ocr(pdf_url, pdf_filename, output_name):
    pdf_path    = RAW_MANUAL_DIR / pdf_filename
    output_file = RAW_MANUAL_DIR / f"{output_name}_ocr.jsonl"

    if output_file.exists() and output_file.stat().st_size > 0:
        print(f"{output_file.name} already exists. Skipping.")
        return

    if not pdf_path.exists():
        print(f"Downloading {pdf_filename}...")
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
            "Referer": "https://archive.org"
        }
        r = requests.get(pdf_url, headers=headers, stream=True)
        if r.status_code == 200:
            with open(pdf_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            print("Download complete.")
        else:
            print(f"Failed. HTTP {r.status_code}")
            return

    print(f"Running OCR on {pdf_path.name}...")
    doc = pymupdf.open(pdf_path)
    saved = 0
    with open(output_file, "w", encoding="utf-8") as fh:
        for pnum in tqdm(range(len(doc)), desc=f"OCR {output_name}"):
            page = doc.load_page(pnum)
            pix  = page.get_pixmap(dpi=150)
            img  = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img, lang="tel").strip()
            if text and len(text) > 100:
                fh.write(json.dumps({"text": text, "page": pnum + 1,
                                     "source": f"ocr_{output_name}"}, ensure_ascii=False) + "\n")
                saved += 1
    print(f"OCR done. Saved {saved} pages → {output_file.name}")

# ── Bharatiya Chitrakala (Telugu art history) ──────────────────────────────
process_pdf_ocr(
    pdf_url="https://archive.org/download/BharatiyaChitrakala/BharatiyaChitrakala.pdf",
    pdf_filename="bharatiya_chitrakala.pdf",
    output_name="bharatiya_chitrakala",
)

# ── Navarasa Kadambari (classical stories) ─────────────────────────────────
process_pdf_ocr(
    pdf_url="https://archive.org/download/NavaraSakadambari/NavaraSakadambari.pdf",
    pdf_filename="navarasa_kadambari.pdf",
    output_name="navarasa_kadambari",
)

## 4. Telugu Wikisource XML Dump (`telugu_wikisource_manual.jsonl`)

Parses the full Wikisource XML dump — far faster than scraping individual article pages.

In [ ]:
output_file    = RAW_MANUAL_DIR / "telugu_wikisource_manual.jsonl"
dump_url       = "https://dumps.wikimedia.org/tewikisource/latest/tewikisource-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "tewikisource-latest-pages-articles.xml.bz2"
te_skip = ["మూస:", "వర్గం:", "చిత్రం:", "వికీపీడియా:", "సహాయం:", "వాడుకరి:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print("Downloading Wikisource dump (~2 GB)...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
        print("Download complete.")
    print("Parsing XML dump...")
    n = _parse_wiki_xml(dump_compressed, output_file, "telugu_wikisource_dump", te_skip)
    print(f"Done. Saved {n:,} pages → {output_file.name}")

### 4a. Telugu Wikibooks XML Dump (`telugu_wikibooks.jsonl`)

In [ ]:
output_file    = RAW_MANUAL_DIR / "telugu_wikibooks.jsonl"
dump_url       = "https://dumps.wikimedia.org/tewikibooks/latest/tewikibooks-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "tewikibooks-latest-pages-articles.xml.bz2"
te_skip = ["మూస:", "వర్గం:", "చిత్రం:", "వికీపీడియా:", "సహాయం:", "వాడుకరి:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print("Downloading Wikibooks dump...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
    n = _parse_wiki_xml(dump_compressed, output_file, "telugu_wikibooks_dump", te_skip)
    dump_compressed.unlink(missing_ok=True)
    print(f"Done. Saved {n:,} pages → {output_file.name}")

### 4b. Telugu Wikiquote XML Dump (`telugu_wikiquote.jsonl`)

In [ ]:
output_file    = RAW_MANUAL_DIR / "telugu_wikiquote.jsonl"
dump_url       = "https://dumps.wikimedia.org/tewikiquote/latest/tewikiquote-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "tewikiquote-latest-pages-articles.xml.bz2"
te_skip = ["మూస:", "వర్గం:", "చిత్రం:", "వికీపీడియా:", "సహాయం:", "వాడుకరి:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print("Downloading Wikiquote dump...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
    n = _parse_wiki_xml(dump_compressed, output_file, "telugu_wikiquote_dump", te_skip)
    dump_compressed.unlink(missing_ok=True)
    print(f"Done. Saved {n:,} pages → {output_file.name}")

### 4c. Telugu Wiktionary XML Dump (`telugu_wiktionary.jsonl`)

In [ ]:
output_file    = RAW_MANUAL_DIR / "telugu_wiktionary.jsonl"
dump_url       = "https://dumps.wikimedia.org/tewiktionary/latest/tewiktionary-latest-pages-articles.xml.bz2"
dump_compressed = RAW_MANUAL_DIR / "tewiktionary-latest-pages-articles.xml.bz2"
te_skip = ["మూస:", "వర్గం:", "చిత్రం:", "వికీపీడియా:", "సహాయం:", "వాడుకరి:"]

if output_file.exists() and output_file.stat().st_size > 0:
    print(f"{output_file.name} already exists. Skipping.")
else:
    if not dump_compressed.exists():
        print("Downloading Wiktionary dump...")
        req = urllib.request.Request(dump_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dump_compressed, "wb") as f:
            shutil.copyfileobj(r, f)
    n = _parse_wiki_xml(dump_compressed, output_file, "telugu_wiktionary_dump", te_skip)
    dump_compressed.unlink(missing_ok=True)
    print(f"Done. Saved {n:,} entries → {output_file.name}")

## 5. Internet Archive: Vipula Magazine (`telugu_vipula_magazine.jsonl`)

Downloads all 306 issues (~156 MB) of Vipula, a Telugu monthly from Eenadu Group (1978–present), from Internet Archive collection `vipula-eenadu-group`.

In [ ]:
out = RAW_MANUAL_DIR / "telugu_vipula_magazine.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    print("Downloading Vipula Magazine (306 issues) from Internet Archive...")
    files = _ia_metadata("vipula-eenadu-group")
    print(f"  Found {len(files)} txt files")
    n = _ia_download_parallel(files, out, "telugu_vipula_magazine", n_threads=15)
    print(f"Done. Saved {n:,} paragraphs → {out.name}")

## 6. Internet Archive: Telugu Newspapers (`telugu_newspapers.jsonl`)

Downloads OCR text from 6 editions of Andhra Jyothi, Eenadu, and Sakshi newspapers from Internet Archive.

In [ ]:
newspaper_items = [
    ("Andhra-Jyothi-telugu-newspaper-24-08-2017",        "telugu_andhra_jyothi"),
    ("eenadu-all-editions-20-10-2024",                    "telugu_eenadu"),
    ("Sakshi-telugu-newspaper-11-12-2024",                "telugu_sakshi"),
    ("eenadu-all-editions-22-10-2024",                    "telugu_eenadu"),
    ("Sakshi-telugu-newspaper-14-09-2024",                "telugu_sakshi"),
    ("andhra-jyothi-epaper-all-editions-2024-02-25",      "telugu_andhra_jyothi"),
]
out = RAW_MANUAL_DIR / "telugu_newspapers.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    for item_id, tag in newspaper_items:
        files = _ia_metadata(item_id)
        print(f"  {item_id}: {len(files)} files...")
        n = _ia_download_parallel(files, out, tag, n_threads=15)
        print(f"    → {n:,} paragraphs saved")
    print(f"Done → {out.name}")

## 7. Internet Archive: Classical Telugu Texts (`telugu_puranas_classical.jsonl`)

Brahmanda Puranam (11 vols, 1868) and Visnu Puranam from Internet Archive.

In [ ]:
purana_items = [
    ("vaedhoktha-moola-sthambhanamu-and-devanga-puranam-in-brahmanda-puranam-estd-1868-vol-1",
     "telugu_brahmanda_puranam"),
    ("tel_b_0107_d_0313", "telugu_visnu_puranam"),
]
out = RAW_MANUAL_DIR / "telugu_puranas_classical.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    for item_id, tag in purana_items:
        files = _ia_metadata(item_id)
        n = _ia_download_parallel(files, out, tag, n_threads=5)
        print(f"  {item_id}: {n:,} paragraphs saved")
    print(f"Done → {out.name}")

## 8. Internet Archive: Telugu Encyclopedia (`telugu_vignana_sarvasvam.jsonl`)

Vignana Sarvasvam Vol V (Vishwa Sahithi) — a comprehensive Telugu-language encyclopedia.

In [ ]:
out = RAW_MANUAL_DIR / "telugu_vignana_sarvasvam.jsonl"
if out.exists() and out.stat().st_size > 0:
    print(f"Already exists: {out.name}. Skipping.")
else:
    files = _ia_metadata("VignanaSarvasvamVolVVishwaSahithi")
    n = _ia_download_parallel(files, out, "telugu_encyclopedia", n_threads=3)
    print(f"Done. Saved {n:,} paragraphs → {out.name}")

## 9. Dataset Audit

Computes line count, word count, and character count for all collected JSONL files.

## 10. Internet Archive: Telugu Books Bulk OCR (`telugu_ia_books.jsonl`)

Queries the Internet Archive search API for all Telugu-language texts with pre-existing
`_djvu.txt` OCR files and downloads them in parallel — no re-OCR required.

- **Source**: `archive.org/advancedsearch.php?q=language:Telugu AND mediatype:texts`
- Parallelized: 20 metadata threads + 12 download threads
- Resumable: progress saved to `telugu_ia_books.done_ids`
- Filters paragraphs to ≥35% Telugu characters

In [ ]:
import json, ssl, time, threading, urllib.request, urllib.parse, re
from pathlib import Path
from queue import Queue

_TE_RE2 = re.compile(r"[\u0C00-\u0C7F]")
_CTX2   = ssl._create_unverified_context()

def _ia_fetch2(url, retries=3):
    for a in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=_CTX2, timeout=30) as r:
                return r.read()
        except Exception: time.sleep(2 ** a)
    return b""

def run_ia_telugu_books(limit=3000, n_meta=20, n_dl=12):
    out_path  = RAW_MANUAL_DIR / "telugu_ia_books.jsonl"
    done_path = RAW_MANUAL_DIR / "telugu_ia_books.done_ids"
    done = set(done_path.read_text(encoding="utf-8").splitlines()) if done_path.exists() else set()
    print(f"Already processed {len(done):,} items.")

    # Step 1: search IA
    ids, page = [], 1
    while len(ids) < limit:
        url = ("https://archive.org/advancedsearch.php?"
               "q=language%3ATelugu+AND+mediatype%3Atexts"
               f"&fl[]=identifier&output=json&rows=200&page={page}")
        docs = json.loads(_ia_fetch2(url) or b"{}").get("response", {}).get("docs", [])
        if not docs: break
        ids.extend(d["identifier"] for d in docs)
        print(f"  Search page {page}: {len(docs)} items ({len(ids):,} total)")
        page += 1; time.sleep(1)
    ids = ids[:limit]
    new_ids = [i for i in ids if i not in done]
    print(f"New items to check: {len(new_ids):,}")

    # Step 2: find djvu.txt files
    meta_q, dl_q = Queue(), Queue()
    ml = threading.Lock()
    for iid in new_ids: meta_q.put(iid)

    def _mw():
        while not meta_q.empty():
            try: iid = meta_q.get_nowait()
            except: break
            try:
                meta = json.loads(_ia_fetch2(f"https://archive.org/metadata/{iid}") or b"{}")
                for f in meta.get("files", []):
                    nm = f.get("name", "")
                    if nm.endswith("_djvu.txt") or ("_djvu" in nm and nm.endswith(".txt")):
                        dl_q.put((iid, nm)); break
                with ml: done.add(iid)
            except Exception: pass
            time.sleep(0.3)

    mts = [threading.Thread(target=_mw) for _ in range(n_meta)]
    for t in mts: t.start()
    for t in mts: t.join()
    print(f"Items with OCR: {dl_q.qsize():,}")

    # Step 3: download
    lock, total, nd = threading.Lock(), [0], [0]

    def _dw():
        while not dl_q.empty():
            try: iid, fname = dl_q.get_nowait()
            except: break
            url  = f"https://archive.org/download/{iid}/{urllib.parse.quote(fname)}"
            text = (_ia_fetch2(url) or b"").decode("utf-8", errors="ignore")
            rows = []
            for para in text.split("\n\n"):
                para = para.strip()
                if len(para.split()) < 40: continue
                chars = len(para.replace(" ", ""))
                if chars and len(_TE_RE2.findall(para)) / chars >= 0.35:
                    rows.append(json.dumps({"text": para, "source": "telugu_ia_books",
                                             "item_id": iid}, ensure_ascii=False))
            if rows:
                with lock:
                    with open(out_path, "a", encoding="utf-8") as fh:
                        fh.write("\n".join(rows) + "\n")
                    total[0] += len(rows)
            with lock: nd[0] += 1
            if nd[0] % 50 == 0:
                done_path.write_text("\n".join(done), encoding="utf-8")
                print(f"  {nd[0]} items | {total[0]:,} paragraphs saved")
            time.sleep(0.5)

    dts = [threading.Thread(target=_dw) for _ in range(n_dl)]
    for t in dts: t.start()
    for t in dts: t.join()
    done_path.write_text("\n".join(done), encoding="utf-8")
    print(f"Done. {total[0]:,} paragraphs → {out_path.name}")

out_te_ia = RAW_MANUAL_DIR / "telugu_ia_books.jsonl"
if out_te_ia.exists() and out_te_ia.stat().st_size > 0:
    wc = sum(len(json.loads(l).get("text","").split()) for l in open(out_te_ia, encoding="utf-8") if l.strip())
    print(f"Already exists: {out_te_ia.name} — {wc:,} words. Skipping.")
else:
    run_ia_telugu_books(limit=3000)

## 11. Web Scrape: eenadu.net (`telugu_news_eenadu.jsonl`)

Pagination-based scraper for `eenadu.net`, the largest Telugu daily newspaper.

**Article URL pattern**: `/telugu-news/{category}/{slug}/{district-code}/{article-id}`

**Strategy**: Scrape each category's listing pages using `?pageno=N` pagination.
Categories covered: telugu-news, national, movies, sports, business, world, nri, crime,
andhra, telangana, devotional, health, education, special-stories (14 categories × up to 5000 pages).

- Resumable via `telugu_news_eenadu.jsonl.seen` file
- Rate-limited: 1.5s delay + random jitter
- Filters to ≥35% Telugu character ratio

In [ ]:
import re, json, time, random
from bs4 import BeautifulSoup

_TE_RE3 = re.compile(r"[\u0C00-\u0C7F]")

EENADU_CATEGORIES = [
    ("telugu-news",     "https://www.eenadu.net/telugu-news?pageno={page}"),
    ("national",        "https://www.eenadu.net/telugu-news/national?pageno={page}"),
    ("movies",          "https://www.eenadu.net/telugu-news/movies?pageno={page}"),
    ("sports",          "https://www.eenadu.net/telugu-news/sports?pageno={page}"),
    ("business",        "https://www.eenadu.net/telugu-news/business?pageno={page}"),
    ("world",           "https://www.eenadu.net/telugu-news/world?pageno={page}"),
    ("andhra",          "https://www.eenadu.net/telugu-news/andhra?pageno={page}"),
    ("telangana",       "https://www.eenadu.net/telugu-news/telangana?pageno={page}"),
    ("devotional",      "https://www.eenadu.net/telugu-news/devotional?pageno={page}"),
    ("health",          "https://www.eenadu.net/telugu-news/health?pageno={page}"),
    ("education",       "https://www.eenadu.net/telugu-news/education?pageno={page}"),
    ("special-stories", "https://www.eenadu.net/telugu-news/special-stories?pageno={page}"),
]
EENADU_SELECTORS = [
    "div.full-details", "div[class*='story-detail']",
    "div.news-content", "div.artical-details", "article",
]

def _scrape_eenadu():
    out_path  = RAW_MANUAL_DIR / "telugu_news_eenadu.jsonl"
    seen_path = RAW_MANUAL_DIR / "telugu_news_eenadu.jsonl.seen"
    seen = set(seen_path.read_text(encoding="utf-8").splitlines()) if seen_path.exists() else set()
    print(f"eenadu: {len(seen):,} URLs already seen")

    import requests as _req
    sess = _req.Session()
    sess.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Chrome/124.0.0.0",
        "Accept-Language": "te,en;q=0.9", "Referer": "https://www.eenadu.net/",
    })
    saved = 0

    def _is_te(text):
        chars = len(text.replace(" ","").replace("\n",""))
        return chars > 0 and len(_TE_RE3.findall(text)) / chars >= 0.35

    def _extract(html):
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script","style","nav","footer","header","aside","noscript"]): tag.decompose()
        for sel in EENADU_SELECTORS:
            try:
                el = soup.select_one(sel)
                if el:
                    t = el.get_text(separator="\n", strip=True)
                    if len(t.split()) >= 80: return t
            except: pass
        best, bn = "", 0
        for tag in soup.find_all(["article","main","div"]):
            t = tag.get_text(separator="\n", strip=True)
            if len(t) > bn and len(t.split()) >= 80: best, bn = t, len(t)
        return best

    def _get_links(html):
        soup = BeautifulSoup(html, "html.parser")
        links = set()
        for a in soup.find_all("a", href=True):
            h = a["href"]
            if not h.startswith("http"): h = "https://www.eenadu.net" + h
            if re.search(r"/telugu-news/[^/]+/[^/]+/\d{4}/\d+", h):
                links.add(h.split("?")[0])
        return links

    for cat, tmpl in EENADU_CATEGORIES:
        empty = 0
        for page in range(1, 5001):
            url = tmpl.format(page=page)
            try:
                time.sleep(1.5 + random.uniform(0, 0.5))
                r = sess.get(url, timeout=20)
                if r.status_code == 404: print(f"  {cat} p{page}: 404 — done"); break
                if r.status_code != 200: time.sleep(5); continue
                links = {u for u in _get_links(r.text) if u not in seen}
                if not links:
                    empty += 1
                    if empty >= 5: print(f"  {cat}: 5 empty pages — done"); break
                    continue
                empty = 0
                for art_url in links:
                    seen.add(art_url)
                    try:
                        time.sleep(1.5 + random.uniform(0, 0.4))
                        ar = sess.get(art_url, timeout=20)
                        if ar.status_code != 200: continue
                        text = _extract(ar.text)
                        if text and _is_te(text):
                            with open(out_path, "a", encoding="utf-8") as fh:
                                fh.write(json.dumps({"text": text, "url": art_url,
                                    "source": "telugu_news_eenadu"}, ensure_ascii=False) + "\n")
                            saved += 1
                    except: pass
                if page % 100 == 0:
                    seen_path.write_text("\n".join(seen), encoding="utf-8")
                    print(f"  {cat} p{page} | saved={saved:,}")
            except Exception as e:
                print(f"  Error {url}: {e}"); time.sleep(10)
        seen_path.write_text("\n".join(seen), encoding="utf-8")

    print(f"eenadu done: {saved:,} articles")

out_eenadu = RAW_MANUAL_DIR / "telugu_news_eenadu.jsonl"
if out_eenadu.exists() and out_eenadu.stat().st_size > 0:
    wc = sum(len(json.loads(l).get("text","").split()) for l in open(out_eenadu, encoding="utf-8") if l.strip())
    print(f"Already exists: {out_eenadu.name} — {wc:,} words. Skipping.")
else:
    _scrape_eenadu()

## 12. Web Scrape: andhrajyothy.com (`telugu_news_andhrajyothy.jsonl`)

Pagination-based scraper for `andhrajyothy.com` — category and district pages.

**Strategy**: Crawls `/page/N` paginated category listings (AP, Telangana, politics, business, etc.)
and district news, extracts unique article links, downloads, and processes them.

- Resumable via `telugu_news_andhrajyothy.jsonl.seen` file
- Filters to ≥35% Telugu characters
- Delay: 1.0s + random jitter

In [ ]:
import re, json, time, random
from bs4 import BeautifulSoup

_AJ_RE = re.compile(r"[\u0C00-\u0C7F]")

AJ_CATEGORIES = [
    ("andhra-pradesh", "https://www.andhrajyothy.com/andhra-pradesh/page/{page}"),
    ("telangana",      "https://www.andhrajyothy.com/telangana/page/{page}"),
    ("national",       "https://www.andhrajyothy.com/national/page/{page}"),
    ("politics",       "https://www.andhrajyothy.com/politics/page/{page}"),
    ("sports",         "https://www.andhrajyothy.com/sports/page/{page}"),
    ("business",       "https://www.andhrajyothy.com/business/page/{page}"),
    ("editorial",      "https://www.andhrajyothy.com/editorial/page/{page}"),
    ("nri",            "https://www.andhrajyothy.com/nri/page/{page}"),
    ("ap-ananthapuram",  "https://www.andhrajyothy.com/andhra-pradesh/ananthapuram/page/{page}"),
    ("ap-guntur",        "https://www.andhrajyothy.com/andhra-pradesh/guntur/page/{page}"),
    ("ap-visakhapatnam", "https://www.andhrajyothy.com/andhra-pradesh/visakhapatnam/page/{page}"),
    ("ap-east-godavari", "https://www.andhrajyothy.com/andhra-pradesh/east-godavari/page/{page}"),
    ("ap-nellore",       "https://www.andhrajyothy.com/andhra-pradesh/nellore/page/{page}"),
    ("ap-chittoor",      "https://www.andhrajyothy.com/andhra-pradesh/chittoor/page/{page}"),
    ("ts-hyderabad",   "https://www.andhrajyothy.com/telangana/hyderabad/page/{page}"),
    ("ts-warangal",    "https://www.andhrajyothy.com/telangana/warangal/page/{page}"),
    ("ts-karimnagar",  "https://www.andhrajyothy.com/telangana/karimnagar/page/{page}"),
]

def _scrape_andhrajyothy():
    out_path  = RAW_MANUAL_DIR / "telugu_news_andhrajyothy.jsonl"
    seen_path = RAW_MANUAL_DIR / "telugu_news_andhrajyothy.jsonl.seen"
    seen = set(seen_path.read_text(encoding="utf-8").splitlines()) if seen_path.exists() else set()
    print(f"andhrajyothy: {len(seen):,} URLs already seen")

    import requests as _req
    sess = _req.Session()
    sess.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Chrome/124.0.0.0",
        "Accept-Language": "te,en;q=0.9", "Referer": "https://www.andhrajyothy.com/",
    })
    saved = 0

    def _is_te(text):
        chars = len(text.replace(" ","").replace("\n",""))
        return chars > 0 and len(_AJ_RE.findall(text)) / chars >= 0.35

    def _extract(html):
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script","style","nav","footer","header","aside","noscript"]): tag.decompose()
        for sel in ["div[class*='detail']", "div.story-content", "article", "main"]:
            try:
                el = soup.select_one(sel)
                if el:
                    t = el.get_text(separator="\n", strip=True)
                    if len(t.split()) >= 80: return t
            except: pass
        best, bn = "", 0
        for tag in soup.find_all(["article","main","div"]):
            t = tag.get_text(separator="\n", strip=True)
            if len(t) > bn and len(t.split()) >= 80: best, bn = t, len(t)
        return best

    def _get_links(html):
        soup = BeautifulSoup(html, "html.parser")
        links = set()
        for a in soup.find_all("a", href=True):
            h = a["href"]
            if not h.startswith("http"): h = "https://www.andhrajyothy.com" + h
            if h.endswith(".html") and re.search(r"/\d{4}/", h):
                links.add(h.split("?")[0])
        return links

    for cat, tmpl in AJ_CATEGORIES:
        empty = 0
        for page in range(1, 36):
            url = tmpl.format(page=page)
            print(f"Fetching {cat} page {page}...")
            try:
                time.sleep(1.0 + random.uniform(0, 0.4))
                r = sess.get(url, timeout=20)
                if r.status_code == 404: print(f"  {cat} p{page}: 404 — done"); break
                if r.status_code != 200: time.sleep(5); continue
                links = {u for u in _get_links(r.text) if u not in seen}
                if not links:
                    empty += 1
                    if empty >= 5: print(f"  {cat}: 5 empty pages — done"); break
                    continue
                empty = 0
                for art_url in links:
                    seen.add(art_url)
                    try:
                        time.sleep(1.0 + random.uniform(0, 0.3))
                        ar = sess.get(art_url, timeout=20)
                        if ar.status_code != 200: continue
                        ar.encoding = "utf-8"
                        text = _extract(ar.text)
                        if text and _is_te(text):
                            with open(out_path, "a", encoding="utf-8") as fh:
                                fh.write(json.dumps({"text": text, "url": art_url,
                                    "source": "telugu_news_andhrajyothy"}, ensure_ascii=False) + "\n")
                            saved += 1
                    except: pass
                if page % 5 == 0:
                    seen_path.write_text("\n".join(seen), encoding="utf-8")
                    print(f"  {cat} p{page} | saved={saved:,}")
            except Exception as e:
                print(f"  Error {url}: {e}"); time.sleep(10)
        seen_path.write_text("\n".join(seen), encoding="utf-8")

    print(f"andhrajyothy done: {saved:,} articles")

out_aj = RAW_MANUAL_DIR / "telugu_news_andhrajyothy.jsonl"
if out_aj.exists() and out_aj.stat().st_size > 0:
    wc = sum(len(json.loads(l).get("text","").split()) for l in open(out_aj, encoding="utf-8") if l.strip())
    print(f"Already exists: {out_aj.name} — {wc:,} words. Skipping.")
else:
    _scrape_andhrajyothy()

In [ ]:
import glob

all_files = sorted(RAW_MANUAL_DIR.glob("*.jsonl"))
print(f"{'File':<45}  {'Lines':>8}  {'Words':>12}  {'Chars':>14}")
print("-" * 85)
grand_lines = grand_words = grand_chars = 0
for fpath in all_files:
    lines = words = chars = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                text = json.loads(line).get("text", "")
            except Exception:
                text = line
            lines += 1
            words += len(text.split())
            chars += len(text)
    grand_lines += lines
    grand_words += words
    grand_chars += chars
    print(f"{fpath.name:<45}  {lines:>8,}  {words:>12,}  {chars:>14,}")
print("-" * 85)
print(f"{'TOTAL':<45}  {grand_lines:>8,}  {grand_words:>12,}  {grand_chars:>14,}")